# Structured Generation

In [24]:
from outlines import from_openai
from pydantic import BaseModel, Field
from openai import OpenAI
import json

# Simple Pydantic structure
class DadJoke(BaseModel):
    setup: str = Field(..., description="Joke setup")
    punchline: str = Field(..., description="Punchline")

# Ollama OpenAI-compatible endpoint
openai_client = OpenAI(
    api_key="ollama",  # Dummy key for local Ollama
    base_url="http://localhost:11434/v1"  # Ollama's OpenAI-compatible API
)

model = from_openai(openai_client, model_name="llama3.2")

def generate_dad_joke(topic: str):
    """Generate perfect dad jokes with guaranteed JSON structure"""
    prompt = f"Tell a perfect data joke about {topic}."
    
    # from_openai handles structured generation automatically
    joke = model(prompt, DadJoke)
    return joke

# Demo
topic = input("Dad joke about? ")
joke = generate_dad_joke(topic)
joke = json.loads(joke)
#print(f"Q: {joke["setup"]}")
#print(f"A: {joke["punchline"]} 😂")
print("Question: {}".format(joke['setup']))
print("Answer: {}".format(joke['punchline']))

Dad joke about?  soccer


Question: Why did the soccer player bring a ladder to 'data practice'?
Answer: Because he wanted to take his game to an upper percentile!


# Dad Joke Agent 

In [50]:
from outlines import from_openai
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import Optional, List, Dict
import json

# Dad joke structure
class DadJoke(BaseModel):
    setup: str = Field(..., description="Joke setup or question")
    punchline: str = Field(..., description="Funny punchline or answer")

# Calculator tool schema (explicit fields, now includes division)
class ToolCall(BaseModel):
    name: str = Field(
        "calculator",
        description="Use this tool to perform simple arithmetic operations: add, subtract, multiply, or divide."
    )
    x: float = Field(..., description="First number for the operation")
    y: float = Field(..., description="Second number for the operation")
    operation: str = Field(
        ..., 
        description="The mathematical operation to perform: one of 'add', 'subtract', 'multiply', 'divide'."
    )

# Agent reasoning schema
class AgentAction(BaseModel):
    thought: str = Field(..., description="Internal reasoning about what to do next.")
    action: Optional[ToolCall] = Field(None, description="Tool call if calculation is needed.")
    final_joke: Optional[DadJoke] = Field(None, description="Final dad joke when ready.")

# Ollama/OpenAI-compatible model
openai_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
model = from_openai(openai_client, model_name="qwen3:latest")

# Calculator tool implementation (now supports division)
def calculator_tool(x: float, y: float, operation: str) -> str:
    """Perform simple math operations including division."""
    try:
        if operation == "add":
            return str(x + y)
        elif operation == "subtract":
            return str(x - y)
        elif operation == "multiply":
            return str(x * y)
        elif operation == "divide":
            if y == 0:
                return "Error: Division by zero"
            return str(x / y)
        else:
            return f"Error: Unknown operation '{operation}'"
    except Exception as e:
        return f"Error: {e}"

# Main agent loop
def generate_dad_joke_agent(topic: str, max_steps: int = 3):
    context = f"""
You are a REASONING-ENABLED Dad Joke Agent writing a math-themed joke about the number '{topic}'.

**Your Objective:**
Create a funny dad joke that includes math related to this number.
The math in the joke **must be correct** — you can use the calculator tool to verify or compute results.

**Available Tool:**
- calculator(x, y, operation): Performs 'add', 'subtract', 'multiply', or 'divide'.

**Reasoning Guidelines:**
1. Think through what kind of math fact or operation fits a joke about {topic}.
2. If you need a computation, call the calculator tool with the appropriate parameters.
3. After receiving an observation, USE that information in your next reasoning step — do NOT repeat the same calculation unless a new one is necessary.
4. Once the math is verified or computed, produce the final dad joke using the `final_joke` field.

**Your output each turn must be one of the following:**
- A reasoning step (`thought`) and a tool call (`action`).
- A reasoning step (`thought`) and the final joke (`final_joke`).

Example Thought Flow:
- Thought: “Maybe 8 divided by 2 equals 4 could fit a joke.”
- Action → calculator(8, 2, 'divide')
- Observation: Calculator result: 4
- Thought: “Nice, that’s correct! I’ll use 4 as the punchline number.”
- Final_joke → (“Why did 8 break up with 2? Because it couldn’t handle the division.”)

Be concise, logical, and make the math part funny.
"""

    messages: List[Dict] = [{"role": "user", "content": context}]
    observations = []

    for step in range(max_steps):
        print(f"\n--- Step {step + 1} ---")
        
        prompt = "\n".join([f"{m['role']}: {m['content']}" for m in messages[-4:]])
        if observations:
            prompt += f"\nLatest Observation: {observations[-1]}"

        # Generate structured response
        try:
            response = model(prompt, AgentAction)
            action_obj = (
                response if isinstance(response, AgentAction)
                else AgentAction.model_validate_json(response)
            )
        except Exception as e:
            print(f"Parse error: {e}")
            return {"error": "Generation failed"}

        print(f"Thought: {action_obj.thought}")

        # Handle tool calls
        if action_obj.action:
            tool = action_obj.action
            print(f"Tool Call: {tool.name}({tool.x}, {tool.y}, operation={tool.operation})")
            result = calculator_tool(tool.x, tool.y, tool.operation)
            
            observation = f"Calculator result: {result}"
            observations.append(observation)
            
            messages.extend([
                {"role": "assistant", "content": f"Action: calculator({tool.x}, {tool.y}, '{tool.operation}')"},
                {"role": "system", "content": observation}
            ])
        
        elif action_obj.final_joke:
            return action_obj.final_joke

        else:
            print("No action or final joke returned.")
            break

    return {"error": f"Max steps ({max_steps}) reached without joke"}


topic = input("Dad joke about the number: ")
result = generate_dad_joke_agent(topic)

if "error" in result:
    print(f"❌ {result['error']}")
else:
    print("\n😂 JOKE:")
    print("Question: {}".format(result.setup))
    print("Answer: {}".format(result.punchline))


Dad joke about the number:  193



--- Step 1 ---
Thought: I need to create a dad joke involving 193 with correct math. Let me verify if 193 is a prime number, as that's a common math fact.
Tool Call: calculator(193.0, 2.0, operation=divide)

--- Step 2 ---
Thought: The division result of 96.5 is interesting. Maybe use it in a joke about splitting or handling decimals.

😂 JOKE:
Question: Why did 193 split up with 2?
Answer: Because it couldn't handle the 96.5!
